# Aurelius v1.x — base-ablation × trace-scaling sweep
Which clean **base** + our SFT wins, and how far **more verified traces** keep helping. Held-out gym eval, tabled.

**HOW TO RUN:** Upload to **Google Colab**, set **Runtime → A100**, **Run all**. These are notebook cells — don't paste into a terminal. Edit the `--bases` / `--trace_sizes` in the last cell. Each grid cell ≈ one SFT (~30–70 min), so keep the grid small. (Run this *after* v1 ships.)

In [ ]:
# 1. Setup
!git clone --depth 1 -b spike/first-light-readiness https://github.com/S3nna13/Aurelius.git
%cd Aurelius
!pip -q install -U transformers peft accelerate jsonschema pyyaml
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set A100")

## 2. Run the sweep
- **Base ablation:** list bases in `--bases`, one `--trace_sizes`.
- **Trace scaling:** one base, list sizes (e.g. `--trace_sizes 60,150,300`).
Generates traces once per size (reused across bases), SFTs each grid cell, evals held-out, prints a ranked table + saves `sweep_results.json`.

In [ ]:
# EDIT --bases / --trace_sizes
!python docs/training/sweep_v1x.py \
  --bases WeiboAI/VibeThinker-3B,Qwen/Qwen3-8B \
  --trace_sizes 150 \
  --teacher deepseek-ai/DeepSeek-R1-Distill-Qwen-7B \
  --sft_rows 8000 --n_eval 60 --out /content/sweep

### Read it
Top row = the winning base/trace-budget → drop it into the v1 notebook (`--base` in cells 3–5, `--n` in cell 2). If the trace-scaling curve is still rising, more traces are worth it; if flat, the next lever is a **bigger base** or **LayerDelta**.